# Merge Augmented Data
Combina il dataset originale con i campioni sintetici generati (STYLE_GAN_ADA o WGAN-GP), controlla la distribuzione e salva un nuovo `dataset_augmented.h5` pronto per il training.

## Section 1: Monta Google Drive
Necessario per leggere l'HDF5 originale e i sintetici su Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Section 2: Config path e scelta GAN
Imposta i percorsi e scegli quale set sintetico usare (`GAN_TYPE = 'CGAN'` o `GAN_TYPE = 'WGAN_GP'`).

In [ ]:
import numpy as np
import h5py, os
import matplotlib.pyplot as plt

ORIG_PATH = '/content/drive/MyDrive/final_scripts/dataset/dataset.h5'
GAN_TYPE = 'STYLE_GAN'  # 'STYLE_GAN' or 'WGAN_GP'

if GAN_TYPE == 'STYLE_GAN':
    SYN_DIR = '/content/drive/MyDrive/STYLE_GAN_augmentation'
else:
    SYN_DIR = '/content/drive/MyDrive/WGAN_GP_augmentation'

print('Using synthetic dir:', SYN_DIR)

## Section 3: Carica dataset originale
Legge train/val dall'HDF5 di origine per avere riferimento e class names.

In [ ]:
# Load original
with h5py.File(ORIG_PATH, 'r') as f:
    X_train = np.array(f['X_train'])
    y_train = np.array(f['y_train'])
    X_val = np.array(f['X_val'])
    y_val = np.array(f['y_val'])
    class_names = [c.decode('utf-8') for c in f['class_names']]
print('Original shapes', X_train.shape, y_train.shape, X_val.shape, y_val.shape)

## Section 4: Carica sintetici e rimappa etichette
Carica immagini/etichette generate, rimappa le etichette locali alle classi globali e normalizza in [0,1].

In [ ]:
# Load synthetic
synth_images = np.load(os.path.join(SYN_DIR, 'synthetic_images.npy'))
synth_labels = np.load(os.path.join(SYN_DIR, 'synthetic_labels.npy'))
rare_idx = np.load(os.path.join(SYN_DIR, 'rare_class_indices.npy'))
print('Synthetic', synth_images.shape, synth_labels.shape)

# Map synthetic labels (local) back to global class ids
label_map_rev = {local: global_id for local, global_id in enumerate(rare_idx)}
synth_labels_global = np.array([label_map_rev[v] for v in synth_labels])
# Normalize back to [0,1] float32 for training
synth_images_float = synth_images.astype('float32') / 255.0
print('Mapped labels shape', synth_labels_global.shape)

## Section 5: Merge e shuffle
Concatena train originale + sintetici, poi mescola per evitare ordini di blocco.

In [ ]:
# Concatenate with original train split
X_train_new = np.concatenate([X_train.astype('float32')/255.0, synth_images_float])
y_train_new = np.concatenate([y_train, synth_labels_global])
# Keep val/test unchanged
print('New train shape', X_train_new.shape, y_train_new.shape)

# Shuffle
idx = np.random.permutation(len(X_train_new))
X_train_new = X_train_new[idx]
y_train_new = y_train_new[idx]

## Section 6: Controllo distribuzione
Verifica i conteggi di classe dopo il merge per valutare il riequilibrio.

In [ ]:
# Quick distribution check
import collections
cnt = collections.Counter(y_train_new)
print('Class counts after merge:')
for i, name in enumerate(class_names):
    print(f'{name:10s}: {cnt[i]}')
plt.bar(class_names, [cnt[i] for i in range(len(class_names))])
plt.xticks(rotation=45)
plt.show()

## Section 7: Salvataggio dataset fuso
Salva `dataset_augmented.h5` con train (originale+synthetic) e val invariato.

In [ ]:
# Save new dataset
OUT_PATH = '/content/drive/MyDrive/final_scripts/dataset/dataset_augmented.h5'
with h5py.File(OUT_PATH, 'w') as f:
    f.create_dataset('X_train', data=X_train_new, compression='gzip')
    f.create_dataset('y_train', data=y_train_new, compression='gzip')
    f.create_dataset('X_val', data=X_val.astype('float32')/255.0, compression='gzip')
    f.create_dataset('y_val', data=y_val, compression='gzip')
    f.create_dataset('class_names', data=np.array(class_names, dtype='S'))
print('Saved', OUT_PATH)